# Worked example: request flow through NGINX and FastAPI

Run this notebook one cell at a time to follow an embeddings request through:

```text
Notebook client -> NGINX -> FastAPI gateway -> embeddings service -> FastAPI gateway -> NGINX -> notebook client
```

The notebook also demonstrates where invalid API keys, invalid JSON payloads, and oversized requests are rejected.

## 1. Locate the example and define helpers

The HTTP helper uses only Python's standard library. The Docker helper runs commands from the folder containing `docker-compose.yml`.

In [6]:
from pathlib import Path
from pprint import pprint
import json
import subprocess
import urllib.error
import urllib.request


def find_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "system-design" / "01-api-gateway",
    ]
    for candidate in candidates:
        if (candidate / "docker-compose.yml").exists():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook from the repository root or 01-api-gateway folder")


PROJECT_DIR = find_project_dir()
BASE_URL = "http://localhost:8080"


def compose(*args, check=True):
    result = subprocess.run(
        ["docker", "compose", *args],
        cwd=PROJECT_DIR,
        text=True,
        capture_output=True,
        check=check,
    )
    print(result.stdout or result.stderr)
    return result


def api_request(method, path, *, headers=None, body=None):
    request_headers = dict(headers or {})
    data = None
    if body is not None:
        data = json.dumps(body).encode("utf-8")
        request_headers.setdefault("Content-Type", "application/json")

    request = urllib.request.Request(
        f"{BASE_URL}{path}",
        data=data,
        headers=request_headers,
        method=method,
    )

    try:
        response = urllib.request.urlopen(request)
    except urllib.error.HTTPError as error:
        response = error

    raw_body = response.read().decode("utf-8")
    try:
        parsed_body = json.loads(raw_body)
    except json.JSONDecodeError:
        parsed_body = raw_body

    result = {
        "status": response.status,
        "server": response.headers.get("Server"),
        "x_request_id": response.headers.get("X-Request-ID"),
        "body": parsed_body,
    }
    pprint(result, sort_dicts=False)
    return result


print(f"Project directory: {PROJECT_DIR}")

Project directory: /Users/Minh/repos/ai-engineering/system-design/01-api-gateway


## 2. Start the complete stack

Only NGINX publishes a host port. The FastAPI gateway and application services remain private inside the Docker network.

In [7]:
compose("up", "--build", "-d")
compose("ps")

#1 [01-api-gateway_gateway internal] load build definition from Dockerfile
#1 transferring dockerfile:
#1 transferring dockerfile: 32B 0.0s done
#1 DONE 0.1s

#2 [01-api-gateway_embeddings-service internal] load build definition from Dockerfile
#2 transferring dockerfile: 32B done
#2 DONE 0.0s

#3 [01-api-gateway_inference-service internal] load build definition from Dockerfile
#3 transferring dockerfile: 32B done
#3 DONE 0.0s

#4 [01-api-gateway_gateway internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [01-api-gateway_inference-service internal] load .dockerignore
#5 transferring context: 2B done
#5 DONE 0.0s

#6 [01-api-gateway_embeddings-service internal] load .dockerignore
#6 transferring context: 2B 0.0s done
#6 DONE 0.1s

#7 [01-api-gateway_inference-service internal] load metadata for docker.io/library/python:3.12-slim
#7 ...

#8 [auth] library/python:pull token for registry-1.docker.io
#8 DONE 0.0s

#7 [01-api-gateway_embeddings-service internal] lo

CompletedProcess(args=['docker', 'compose', 'ps'], returncode=0, stdout='NAME                                  COMMAND                  SERVICE              STATUS              PORTS\n01-api-gateway-embeddings-service-1   "uvicorn app.embeddi…"   embeddings-service   running             8001/tcp\n01-api-gateway-gateway-1              "uvicorn app.gateway…"   gateway              running (healthy)   8080/tcp\n01-api-gateway-inference-service-1    "uvicorn app.inferen…"   inference-service    running             8002/tcp\n01-api-gateway-nginx-1                "/docker-entrypoint.…"   nginx                running             80/tcp, 0.0.0.0:8080->8080/tcp\n', stderr='')

Expected topology:

- `nginx` publishes `localhost:8080`.
- `gateway` exposes port `8080` only inside Docker.
- `embeddings-service` exposes port `8001` only inside Docker.
- `inference-service` exposes port `8002` only inside Docker.

## 3. Health request: client -> NGINX -> FastAPI

NGINX receives the public request and proxies it to FastAPI. The `Server` header confirms that the response returned through NGINX. Because the client does not provide an `X-Request-ID`, NGINX creates one.

In [8]:
health = api_request("GET", "/health")

{'status': 200,
 'server': 'nginx/1.28.3',
 'x_request_id': 'f241488bd3afdf21f9babb4e8fe38806',
 'body': {'status': 'ok', 'service': 'api-gateway'}}


## 4. Successful embeddings request through every component

This request supplies a known request ID so its propagation is easy to see.

1. NGINX checks the 1 MB body limit and per-IP rate limit.
2. NGINX preserves `worked-example-123` and proxies the request to `gateway:8080`.
3. FastAPI validates `X-API-Key` and the JSON body.
4. FastAPI sends `{"input": ...}` and the request ID to `embeddings-service:8001/internal/embed`.
5. The embeddings service returns a toy vector and the same request ID.
6. FastAPI wraps the internal response in the public response shape.
7. NGINX returns the response and adds the request ID response header.

In [9]:
request_id = "worked-example-123"

embedding_response = api_request(
    "POST",
    "/v1/embeddings",
    headers={
        "X-API-Key": "dev-key",
        "X-Request-ID": request_id,
    },
    body={"input": "nginx fronts fastapi"},
)

assert embedding_response["status"] == 200
assert embedding_response["server"].startswith("nginx/")
assert embedding_response["x_request_id"] == request_id
assert embedding_response["body"]["request_id"] == request_id
assert embedding_response["body"]["data"]["request_id"] == request_id

print("\nThe request ID survived the complete round trip.")

{'status': 200,
 'server': 'nginx/1.28.3',
 'x_request_id': 'worked-example-123',
 'body': {'request_id': 'worked-example-123',
          'object': 'embedding',
          'data': {'request_id': 'worked-example-123',
                   'model': 'toy-hash-embedding',
                   'embedding': [0.3843,
                                 0.149,
                                 0.7882,
                                 0.0039,
                                 0.1529,
                                 0.9608,
                                 0.5843,
                                 0.5412],
                   'dimensions': 8}}}

The request ID survived the complete round trip.


## 5. Prove the embeddings service is private

The embeddings service uses Docker Compose `expose`, but it does not publish a host `ports` mapping. Therefore:

- Calling `localhost:8001` from this notebook should fail because the port is not published to the host.
- Calling `embeddings-service:8001` from inside the gateway container should succeed because both containers share Docker's private network.

In [10]:
# This runs on the host machine where the notebook kernel is running.
# It should fail because embeddings-service does not publish port 8001 to localhost.
try:
    urllib.request.urlopen("http://localhost:8001/health", timeout=2)
    raise AssertionError("Unexpectedly reached the private embeddings service from the host")
except urllib.error.URLError as error:
    print(f"Expected host-side failure: {error}")

Expected host-side failure: <urlopen error [Errno 61] Connection refused>


`docker compose exec gateway ...` executes the supplied command **inside the already-running gateway container**. From there, Docker DNS resolves `embeddings-service` and the private port `8001` is reachable.

In [11]:
# Equivalent shell command:
# docker compose exec -T gateway python -c "..."
#
# `exec -T gateway` means: run the remaining command inside the gateway container.
compose(
    "exec", "-T", "gateway", "python", "-c",
    "import urllib.request; print(urllib.request.urlopen('http://embeddings-service:8001/health').read().decode())",
)

{"status":"ok","service":"embeddings"}



CompletedProcess(args=['docker', 'compose', 'exec', '-T', 'gateway', 'python', '-c', "import urllib.request; print(urllib.request.urlopen('http://embeddings-service:8001/health').read().decode())"], returncode=0, stdout='{"status":"ok","service":"embeddings"}\n', stderr='')

## 6. Authentication failure: rejected by FastAPI

This request passes NGINX's transport-level checks, but FastAPI rejects it before calling the embeddings service.

In [12]:
unauthorized = api_request(
    "POST",
    "/v1/embeddings",
    headers={"X-API-Key": "wrong-key"},
    body={"input": "this must not reach the embeddings service"},
)

assert unauthorized["status"] == 401

{'status': 401,
 'server': 'nginx/1.28.3',
 'x_request_id': '32109873abf99d61b296fe302eafd4f3',
 'body': {'detail': 'Missing or invalid API key'}}


## 7. Schema failure: rejected by FastAPI validation

The body is below NGINX's 1 MB limit, but its `input` field exceeds FastAPI's 2,000-character application limit.

In [13]:
invalid_payload = api_request(
    "POST",
    "/v1/embeddings",
    headers={"X-API-Key": "dev-key"},
    body={"input": "x" * 2_001},
)

assert invalid_payload["status"] == 422

{'status': 422,
 'server': 'nginx/1.28.3',
 'x_request_id': '83f2fb76e8c69b644cf17b68eb720690',
 'body': {'detail': [{'type': 'string_too_long',
                      'loc': ['body', 'input'],
                      'msg': 'String should have at most 2000 characters',
                      'input': 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

## 8. Oversized request: rejected by NGINX

This body exceeds NGINX's `client_max_body_size 1m`. NGINX returns `413` without sending the request to FastAPI.

In [14]:
oversized = api_request(
    "POST",
    "/v1/embeddings",
    headers={"X-API-Key": "dev-key"},
    body={"input": "x" * 1_100_000},
)

assert oversized["status"] == 413

{'status': 413,
 'server': 'nginx/1.28.3',
 'x_request_id': '2b723a8d5d55f392e544c46d8ed9c9a3',
 'body': '<html>\r\n'
         '<head><title>413 Request Entity Too Large</title></head>\r\n'
         '<body>\r\n'
         '<center><h1>413 Request Entity Too Large</h1></center>\r\n'
         '<hr><center>nginx/1.28.3</center>\r\n'
         '</body>\r\n'
         '</html>\r\n'}


## 9. Inspect recent container logs

These logs show which public requests reached NGINX and FastAPI. The oversized request appears in NGINX logs but should not appear as a FastAPI request.

In [15]:
compose("logs", "--tail=30", "nginx", "gateway", "embeddings-service")

01-api-gateway-gateway-1  | INFO:     127.0.0.1:39014 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39016 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39018 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39020 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39022 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39024 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39026 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39028 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39030 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39032 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39034 - "GET /health HTTP/1.1" 200 OK
01-api-gateway-gateway-1  | INFO:     127.0.0.1:39036 

CompletedProcess(args=['docker', 'compose', 'logs', '--tail=30', 'nginx', 'gateway', 'embeddings-service'], returncode=0, stdout='01-api-gateway-gateway-1  | INFO:     127.0.0.1:39014 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39016 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39018 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39020 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39022 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39024 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39026 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39028 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39030 - "GET /health HTTP/1.1" 200 OK\n01-api-gateway-gateway-1  | INFO:     127.0.0.1:39032 - "GET /health HTTP/1.1" 200 OK\n

## 10. Stop the stack when finished

Run this cell when you no longer need the example containers.

In [16]:
# Uncomment when you want to stop and remove the containers.
# compose("down")